# Num-Classes Sweep

Sweeps `num_classes ∈ [2, 3, 4, 5]` for each pipeline × dataset, keeping
all other hyperparameters fixed at the best values found by the main
hyperparameter sweep.

**Cell 1** is read-only — inspect the best configs before training.  
**Cell 2** runs (or skips) training — idempotent, safe to rerun.  
**Cells 3–4** collect results and visualise.

In [ ]:
import importlib
import json
import sys
import traceback
from pathlib import Path

import pandas as pd

FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

from baseline.sweeps import (
    num_classes_aggregate,
    train_end_to_end,
    train_end_to_end_2d,
    train_ae_classifier,
    train_ae_classifier_2d,
)
from baseline.sweeps.run_sweep import _prepare_cfg, _result_exists

# Reload to pick up any edits made since the kernel started
for _mod in (num_classes_aggregate, train_end_to_end, train_end_to_end_2d,
             train_ae_classifier, train_ae_classifier_2d):
    importlib.reload(_mod)

LEADERBOARD_PATH = FYP_ROOT / 'baseline' / 'leaderboard.json'
NUM_CLASSES      = [2, 3, 4, 5]

# Maps pipeline name -> (sweep_type_string, runner_function)
PIPELINE_INFO = {
    'end_to_end':       ('end_to_end',       train_end_to_end.run_cv),
    'end_to_end_2d':    ('end_to_end_2d',    train_end_to_end_2d.run_cv),
    'ae_classifier':    ('ae_classifier',    train_ae_classifier.run_cv),
    'ae_classifier_2d': ('ae_classifier_2d', train_ae_classifier_2d.run_cv),
}

print('Setup complete.')

## 1 · Inspect best configs

Shows the best run per pipeline × dataset from the **hyperparameter sweep only**
(outcome-sweep runs with specific `score_name` values are excluded so we pick
the best general-purpose hyperparameters).  These are the hyperparameters that
will be held fixed while `num_classes` is swept.

In [ ]:
with open(LEADERBOARD_PATH) as f:
    lb = json.load(f)

# Best config per (pipeline, dataset) from the hyperparameter sweep.
# score_name=None / 'QL2' means it's a hyperparameter-sweep run;
# outcome-sweep runs have specific names like 'BNCD', 'FA', etc.
seen = {}
for r in lb['ranked']:
    if r.get('needs_rerun') or r.get('f1') is None:
        continue
    if r.get('score_name') not in (None, 'QL2'):
        continue
    key = (r['pipeline'], r.get('dataset', 'brainwear'))
    if key not in seen:
        seen[key] = r

best_records = list(seen.values())
print('Found best configs for {} pipeline x dataset combinations:'.format(len(best_records)))

rows = []
for r in best_records:
    row = {
        'pipeline': r['pipeline'],
        'dataset':  r.get('dataset', 'brainwear'),
        'model':    r.get('model'),
        'f1':       r.get('f1'),
        'bal_acc':  r.get('balanced_accuracy'),
        'run_name': r.get('run_name'),
    }
    for k, v in r.get('config', {}).items():
        row['cfg.' + k] = v
    rows.append(row)

display(
    pd.DataFrame(rows)
      .style
      .format({'f1': '{:.3f}', 'bal_acc': '{:.3f}'}, na_rep='—')
      .set_caption('Best hyperparameter-sweep config per pipeline x dataset')
)

## 2 · Run training

For each best config × each `num_classes` value the cell below:
- **skips** if results already exist (idempotent),
- **runs** training otherwise.

All run directories are collected in `completed_run_dirs` for the analysis cells.

In [ ]:
completed_run_dirs = []  # list of (pipeline, dataset, nc, Path)

for record in best_records:
    pipeline = record['pipeline']
    dataset  = record.get('dataset', 'brainwear')
    cfg_base = record['config']

    # The leaderboard labels both 2D (PNG) and 3D end-to-end / ae_classifier
    # runs with the same pipeline name (it only distinguishes them by the
    # dataset's '_png' suffix). PNG datasets must be routed to the *_2d
    # runners, whose argparse accepts 'brainwear_png' / 'brats_png'.
    pipeline_key = pipeline
    if dataset.endswith('_png') and not pipeline_key.endswith('_2d'):
        pipeline_key += '_2d'

    if pipeline_key not in PIPELINE_INFO:
        print('[skip] unknown pipeline: {}'.format(pipeline_key))
        continue

    sweep_type, runner = PIPELINE_INFO[pipeline_key]
    is_e2e = pipeline_key.startswith('end_to_end')

    for nc in NUM_CLASSES:
        raw = dict(cfg_base)  # copy
        raw['num_classes_sweep'] = True

        if is_e2e:
            raw['num_bins'] = nc
            raw['dataset']  = dataset
        else:
            raw['num_classes'] = nc

        cfg      = _prepare_cfg(sweep_type, dataset, raw)
        existing = _result_exists(sweep_type, cfg)
        tag      = '{}/{}  nc={}'.format(pipeline_key, dataset, nc)

        if existing:
            print('SKIP  {}  ->  {}'.format(tag, existing.parent.name))
            completed_run_dirs.append((pipeline_key, dataset, nc, existing.parent))
        else:
            print('\nRUN   {}'.format(tag))
            try:
                runner(cfg)
                run_dir = _result_exists(sweep_type, cfg)
                if run_dir:
                    completed_run_dirs.append((pipeline_key, dataset, nc, run_dir.parent))
                    print('  done -> {}'.format(run_dir.parent.name))
                else:
                    print('  WARNING: result file not found after run')
            except Exception:
                print('  FAILED:\n{}'.format(traceback.format_exc()))

print('\nTotal run dirs collected: {}'.format(len(completed_run_dirs)))

## 3 · Leaderboard

Builds (and writes) `baseline/num_classes_leaderboard.json`, then displays it as a table.

In [ ]:
importlib.reload(num_classes_aggregate)

lb_nc = num_classes_aggregate.build_leaderboard(write=True)
num_classes_aggregate.print_table(lb_nc)

df_nc = pd.DataFrame(lb_nc['ranked'])
if not df_nc.empty:
    display(
        df_nc[['pipeline', 'dataset', 'model', 'num_classes',
               'balanced_accuracy', 'f1', 'accuracy', 'val_loss', 'run_name']]
          .sort_values(['pipeline', 'dataset', 'num_classes'])
          .style
          .format({'balanced_accuracy': '{:.3f}', 'f1': '{:.3f}',
                   'accuracy': '{:.3f}', 'val_loss': '{:.4f}'}, na_rep='—')
          .set_caption('Num-classes sweep results (balanced accuracy, best hyperparams fixed)')
          .background_gradient(subset=['balanced_accuracy'], cmap='RdYlGn', vmin=0, vmax=1)
    )

## 4 · Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

if 'df_nc' not in dir() or df_nc.empty:
    print('No data to plot yet — run Cell 3 first.')
else:
    groups = list(df_nc.groupby(['pipeline', 'dataset']))
    n_groups = len(groups)
    ncols = min(2, n_groups)
    nrows = (n_groups + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows),
                             squeeze=False)
    axes_flat = axes.flatten()

    random_baselines = {2: 0.5, 3: 1/3, 4: 0.25, 5: 0.2}

    for ax_i, ((pipeline, dataset), grp) in enumerate(groups):
        ax = axes_flat[ax_i]
        grp = grp.sort_values('num_classes')

        ax.plot(grp['num_classes'], grp['balanced_accuracy'],
                marker='o', label='Balanced Acc', color='coral')
        ax.plot(grp['num_classes'], grp['f1'],
                marker='s', label='Macro F1', color='steelblue')

        rand_y = [random_baselines.get(nc, 1.0 / nc) for nc in grp['num_classes']]
        ax.plot(grp['num_classes'], rand_y,
                linestyle='--', color='grey', linewidth=0.8, label='Random baseline')

        ax.set_title('{}\n{}'.format(pipeline, dataset), fontsize=9)
        ax.set_xlabel('num_classes')
        ax.set_ylabel('Score')
        ax.set_xticks(NUM_CLASSES)
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
        ax.legend(fontsize=8)

    for ax_i in range(n_groups, len(axes_flat)):
        axes_flat[ax_i].set_visible(False)

    fig.suptitle('Performance vs num_classes (best hyperparams fixed)', fontsize=11)
    fig.tight_layout()
    plt.show()